# Imports Explanation
These import the Surprise library for building and evaluating recommendation models, Pandas for data manipulation, and Scikit-Learn for accuracy metrics. Note: 'accuracy' is imported from 'surprise' as it's in the accuracy submodule.

In [1]:
from surprise import Dataset, Reader, SVD  # Dataset loads rating data from various formats; Reader specifies the data format and rating scale; SVD is the Singular Value Decomposition algorithm for collaborative filtering.
from surprise.model_selection import train_test_split  # train_test_split splits the dataset into training and testing sets.
from surprise import accuracy  # accuracy provides functions like RMSE to evaluate predictions.
import pandas as pd  # Pandas is used for loading and manipulating the CSV data as DataFrames, making it easy to filter and inspect ratings.

# Data Loading Explanation
Loads the MovieLens ratings data from CSV and converts it into Surprise's Dataset format for model training. The Reader defines the rating scale (1-5).

In [2]:
ratings = pd.read_csv('ratings.csv', sep='\t', names=['user_id', 'movie_id', 'rating', 'timestamp'])  # Loads the ratings CSV into a Pandas DataFrame; sep='\t' specifies tab-separated values; names assigns column headers since the file has none.
reader = Reader(rating_scale=(1, 5))  # Creates a Reader object that tells Surprise the minimum and maximum rating values in the dataset.
data = Dataset.load_from_df(ratings[['user_id', 'movie_id', 'rating']], reader)  # Converts the DataFrame (selecting only user, movie, and rating columns) into a Surprise Dataset object for model compatibility.
print(ratings.head())  # Prints the first 5 rows of the DataFrame to inspect the loaded data, ensuring it looks correct (e.g., user_ids, ratings).

   user_id  movie_id  rating  timestamp
0      196       242       3  881250949
1      186       302       3  891717742
2       22       377       1  878887116
3      244        51       2  880606923
4      166       346       1  886397596


# Model Training Explanation
Splits the dataset into train and test sets, then trains the SVD model on the training data to learn user and item embeddings for recommendation.

In [3]:
trainset, testset = train_test_split(data, test_size=0.2)  # Splits the dataset into 80% training and 20% testing sets for model evaluation; test_size=0.2 specifies the proportion for testing.
model = SVD()  # Initializes the SVD (Singular Value Decomposition) algorithm, which is a matrix factorization technique for collaborative filtering.
model.fit(trainset)  # Fits (trains) the SVD model on the training set, learning the latent factors from user ratings.

# Evaluation and Recommendation Explanation
Evaluates the model using RMSE on the test set, then generates top movie recommendations for a sample user by predicting ratings for unseen movies.

In [4]:
predictions = model.test(testset)  # Uses the trained model to make predictions on the test set; returns a list of prediction objects with actual and estimated ratings.
rmse = accuracy.rmse(predictions)  # Computes the Root Mean Squared Error (RMSE) from the predictions; RMSE measures how close predicted ratings are to actual ones (lower is better).
print(f"RMSE: {rmse:.4f}")  # Prints the RMSE value formatted to 4 decimal places for precision.

# Recommend for a sample user (e.g., user_id = 1)
user_id = 1  # Specifies the user ID for whom to generate recommendations; change to test different users.
all_movies = ratings['movie_id'].unique()  # Gets a list of all unique movie IDs from the ratings data.
rated_movies = ratings[ratings['user_id'] == user_id]['movie_id']  # Filters the movies already rated by this user to avoid recommending them again.
recommendations = []  # Initializes an empty list to store predicted ratings for unseen movies.
for movie_id in all_movies:  # Loops through all movie IDs.
    if movie_id not in rated_movies.tolist():  # Checks if the movie hasn't been rated by the user.
        pred = model.predict(user_id, movie_id).est  # Predicts the rating for this user-movie pair; .est gets the estimated rating.
        recommendations.append((movie_id, pred))  # Appends the movie ID and predicted rating as a tuple.
top_recs = sorted(recommendations, key=lambda x: x[1], reverse=True)[:5]  # Sorts recommendations by predicted rating (descending) and takes top 5.
print("Top Recommendations:", top_recs)  # Prints the top movie IDs and their predicted ratings.

RMSE: 0.9354
RMSE: 0.9354
Top Recommendations: [(408, 5), (318, 4.7490546035508165), (313, 4.681043867856139), (511, 4.602725105616415), (856, 4.570243586684736)]


# Movie Names Mapping Explanation
Loads the movies CSV to map movie IDs to titles, then updates recommendations to show names instead of IDs for better readability.

In [5]:
movies = pd.read_csv('movies.csv', sep='|', usecols=[0, 1], names=['movie_id', 'title'], encoding='latin-1')  # Loads movies CSV into DataFrame; sep='|' for pipe-separated; usecols=[0,1] loads only movie_id and title columns; names assigns headers; encoding='latin-1' handles special characters in titles.
movie_dict = dict(zip(movies['movie_id'], movies['title']))  # Creates a dictionary mapping movie_id to title for fast lookup.
print(movie_dict[50])  # Tests the dictionary with movie ID 50 (should output "Star Wars (1977)").

Star Wars (1977)


# Updated Recommendation Explanation
Re-runs recommendation generation but maps movie IDs to titles using the dictionary for human-readable output.

In [6]:
# Recommend for a sample user (e.g., user_id = 1) with names
user_id = 1  # Specifies the user ID for whom to generate recommendations; change to test different users.
all_movies = ratings['movie_id'].unique()  # Gets a list of all unique movie IDs from the ratings data.
rated_movies = ratings[ratings['user_id'] == user_id]['movie_id']  # Filters the movies already rated by this user to avoid recommending them again.
recommendations = []  # Initializes an empty list to store predicted ratings for unseen movies.
for movie_id in all_movies:  # Loops through all movie IDs.
    if movie_id not in rated_movies.tolist():  # Checks if the movie hasn't been rated by the user.
        pred = model.predict(user_id, movie_id).est  # Predicts the rating for this user-movie pair; .est gets the estimated rating.
        title = movie_dict.get(movie_id, f"Unknown Movie (ID: {movie_id})")  # Gets the movie title from the dictionary; fallback if ID not found.
        recommendations.append((title, pred))  # Appends the title and predicted rating as a tuple.
top_recs = sorted(recommendations, key=lambda x: x[1], reverse=True)[:5]  # Sorts recommendations by predicted rating (descending) and takes top 5.
print("Top Recommendations with Names:", top_recs)  # Prints the top titles and their predicted ratings.

Top Recommendations with Names: [('Close Shave, A (1995)', 5), ("Schindler's List (1993)", 4.7490546035508165), ('Titanic (1997)', 4.681043867856139), ('Lawrence of Arabia (1962)', 4.602725105616415), ('Night on Earth (1991)', 4.570243586684736)]


# Debug Explanation
Performs cross-validation to compute average RMSE across folds, ensuring model performance is consistent and not overfitted to one split.

In [7]:
from surprise.model_selection import cross_validate  # Imports the cross_validate function from Surprise for k-fold validation.
cv_results = cross_validate(model, data, measures=['RMSE'], cv=5, verbose=True)  # Runs 5-fold cross-validation on the full dataset; measures=['RMSE'] specifies the metric; cv=5 sets folds; verbose=True prints progress and results.
print("Average RMSE:", cv_results['test_rmse'].mean())  # Prints the mean RMSE from the 5 folds for overall performance assessment.

Evaluating RMSE of algorithm SVD on 5 split(s).

                  Fold 1  Fold 2  Fold 3  Fold 4  Fold 5  Mean    Std     
RMSE (testset)    0.9272  0.9415  0.9408  0.9351  0.9408  0.9371  0.0055  
Fit time          3.36    3.83    1.23    1.29    1.22    2.19    1.16    
Test time         0.35    1.05    0.17    0.19    0.12    0.37    0.35    
Average RMSE: 0.9370831392447296
